# Chapter 1 - Common Conventions and API Elements of scikit-learn

Reproduksi contoh Chapter 1 **scikit-learn Cookbook, Third Edition**, John Sukup (Packt, 2025), halaman cetak 1–12. Repo penerbit mulai dari Chapter 2; contoh bab ini ditulis dari buku dengan seed tetap, ditambah ilustrasi bagian konseptual. Identitas dan petunjuk menjalankan notebook tercantum pada README.

## Daftar Isi

- [Design Philosophy](#design-philosophy)
- [Understanding Estimators](#understanding-estimators)
- [Transformers and the transform Method](#transformers-and-the-transform-method)
- [Handling Custom Estimators and Transformers](#handling-custom-estimators-and-transformers)
- [Pipelines and Workflow Automation](#pipelines-and-workflow-automation)
- [Common Attributes and Methods](#common-attributes-and-methods)
- [Hyperparameter Tuning with Search Methods](#hyperparameter-tuning-with-search-methods)
- [Working with Metadata: Tags and Routing](#working-with-metadata-tags-and-routing)
- [Best Practices for API Usage](#best-practices-for-api-usage)
- [Pemeriksaan Hasil](#pemeriksaan-hasil)
- [Ringkasan Bab](#ringkasan-bab)

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.utils import Bunch

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
np.random.seed(2024)
pd.set_option('display.max_columns', 12)
pd.set_option('display.max_rows', 25)
plt.rcParams.update({'figure.dpi': 100, 'savefig.bbox': 'tight'})

## Design Philosophy

Antarmuka yang konsisten membuat algoritma dapat diganti tanpa mengubah pola workflow: fit mempelajari parameter, predict menghasilkan respons, dan transform mengubah representasi. Komponen bersifat modular dan dapat digabung. Estimator adalah objek yang dapat fit; tidak semua estimator mempunyai predict. Transformer mempunyai transform, sedangkan predictor mempunyai predict. X biasanya matriks (n_samples, n_features), sedangkan y memuat target.

## Understanding Estimators

LinearRegression mempelajari slope dan intercept dengan least squares. Contoh memisahkan fit pada data lima titik dari predict pada x=6 dan x=7. KMeans memakai fit_predict pada data yang sama untuk clustering; nomor cluster merupakan label arbitrer, bukan urutan kualitas.

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])
model = LinearRegression().fit(X, y)
predictions = model.predict([[6], [7]])
print('Predictions:', predictions)
kmeans = KMeans(n_clusters=2, random_state=2024, n_init=10)
labels = kmeans.fit_predict(X)
print('Cluster labels:', labels)

Predictions: [5.75 6.7 ]
Cluster labels: [0 0 0 1 1]


## Transformers and the transform Method

StandardScaler mempelajari mean dan simpangan baku pada fit, kemudian memakai parameter yang sama pada transform. fit_transform adalah shortcut pada training; fit ulang pada test menyebabkan leakage dan transformasi yang tidak konsisten. Standardisasi memberi mean nol dan varians satu, tetapi tidak membuat bentuk distribusi otomatis normal.

In [3]:
from sklearn.preprocessing import StandardScaler
X_scale = np.array([[1, 2], [3, 4], [5, 6]])
scaler = StandardScaler().fit(X_scale)
X_scaled = scaler.transform(X_scale)
print('Separate fit/transform:', X_scaled)
print('Shortcut:', StandardScaler().fit_transform(X_scale))
print('New points transformed with training statistics:', scaler.transform([[7, 8]]))

Separate fit/transform: [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]
Shortcut: [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]
New points transformed with training statistics: [[2.44948974 2.44948974]]


## Handling Custom Estimators and Transformers

BaseEstimator menyediakan get_params/set_params agar objek dapat di-clone untuk pencarian dan CV. TransformerMixin menyediakan fit_transform. Parameter constructor disimpan tanpa menjalankan pembelajaran; parameter yang dipelajari di fit diberi suffix underscore. check_is_fitted memeriksa status fit, bukan validitas nilai hyperparameter. Contoh tambahan MeanCenterer memusatkan data memakai mean training.

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.utils.validation import check_array, check_is_fitted
class MeanCenterer(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        X = check_array(X)
        self.mean_ = X.mean(axis=0)
        self.n_features_in_ = X.shape[1]
        return self
    def transform(self, X):
        check_is_fitted(self, 'mean_')
        X = check_array(X)
        if X.shape[1] != self.n_features_in_:
            raise ValueError('Number of features differs from training')
        return X - self.mean_
centerer = MeanCenterer().fit(X_scale)
print('Centered training:', centerer.transform(X_scale))
print('Centered new points:', centerer.transform([[7, 8]]))
print('Clone fitted?', hasattr(clone(centerer), 'mean_'))

Centered training: [[-2. -2.]
 [ 0.  0.]
 [ 2.  2.]]
Centered new points: [[4. 4.]]
Clone fitted? False


## Pipelines and Workflow Automation

Pipeline merangkai preprocessing dan model agar urutan fit/predict konsisten. Dalam CV, scaler dipelajari kembali hanya pada training setiap fold. Otomasi preprocessing merupakan bagian workflow; MLOps juga mencakup versi data/model, deployment, monitoring drift, dan retraining. Pipeline saja belum membentuk seluruh sistem produksi.

In [5]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target,
    test_size=0.2, random_state=2024, stratify=iris.target)
pipeline = Pipeline([('scaler', StandardScaler()),
                     ('model', LogisticRegression(max_iter=1000, random_state=2024))])
pipeline.fit(X_train, y_train)
api_pipeline_accuracy = pipeline.score(X_test, y_test)
print('Pipeline test accuracy:', api_pipeline_accuracy)

Pipeline test accuracy: 0.9666666666666667


## Common Attributes and Methods

coef_ dan intercept_ adalah parameter setelah fit. get_params mengembalikan hyperparameter; set_params mengubah konfigurasi dan tidak otomatis refit model. score pada classifier umumnya accuracy, sedangkan regressor umumnya R-squared. R-squared training OLS dengan intercept tidak turun ketika fitur ditambahkan pada data yang sama; aturan itu tidak berlaku otomatis pada skor test atau semua estimator.

In [6]:
print('Coefficients:', model.coef_)
print('Intercept:', model.intercept_)
print('Training R2:', model.score(X, y))
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier()
forest.set_params(n_estimators=100, max_depth=10, random_state=42)
print(forest.get_params())

Coefficients: [0.95]
Intercept: 0.04999999999999938
Training R2: 0.9809782608695652


{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


## Hyperparameter Tuning with Search Methods

GridSearchCV mengevaluasi seluruh kombinasi pada grid. RandomizedSearchCV mengambil sejumlah konfigurasi dari distribusi/rentang, sedangkan successive halving bertahap mengalokasikan sumber daya pada kandidat. Parameter pipeline memakai nama langkah dan dua underscore. Best CV score adalah hasil pencarian pada training, bukan skor test independen.

In [7]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
grid = GridSearchCV(pipeline, {'model__C': [0.1, 1, 10]}, cv=3,
                    scoring='accuracy', error_score='raise')
grid.fit(X_train, y_train)
random_search = RandomizedSearchCV(pipeline, {'model__C': [0.01, 0.1, 1, 10, 100]},
    n_iter=3, cv=3, random_state=2024, scoring='accuracy', error_score='raise')
random_search.fit(X_train, y_train)
print('Grid best:', grid.best_params_, grid.best_score_)
print('Randomized best:', random_search.best_params_, random_search.best_score_)
print('Grid test accuracy:', grid.score(X_test, y_test))

Grid best: {'model__C': 1} 0.9583333333333334
Randomized best: {'model__C': 10} 0.9499999999999998
Grid test accuracy: 0.9666666666666667


## Working with Metadata: Tags and Routing

Estimator tags menjelaskan kemampuan, misalnya kebutuhan target atau dukungan sparse input. Metadata routing menentukan penerusan informasi tambahan seperti sample_weight ke langkah yang memintanya; tags dan routing merupakan dua konsep berbeda. Routing perlu diaktifkan dan dikonfigurasi. Contoh tambahan meneruskan bobot hanya ke model, sementara scaler tidak meminta bobot.

In [8]:
from sklearn import set_config
set_config(enable_metadata_routing=True)
weighted_model = LogisticRegression(max_iter=1000, random_state=2024).set_fit_request(sample_weight=True)
unweighted_scaler = StandardScaler().set_fit_request(sample_weight=False)
weighted_pipeline = Pipeline([('scaler', unweighted_scaler), ('model', weighted_model)])
weights = np.where(y_train == 2, 2.0, 1.0)
weighted_pipeline.fit(X_train, y_train, sample_weight=weights)
print(weighted_pipeline.get_metadata_routing())
print('Estimator type:', weighted_model._estimator_type)
set_config(enable_metadata_routing=False)

{'scaler': {'mapping': [{'caller': 'fit', 'callee': 'fit_transform'}, {'caller': 'fit_transform', 'callee': 'fit_transform'}, {'caller': 'fit_predict', 'callee': 'fit_transform'}, {'caller': 'predict', 'callee': 'transform'}, {'caller': 'predict', 'callee': 'transform'}, {'caller': 'predict_proba', 'callee': 'transform'}, {'caller': 'decision_function', 'callee': 'transform'}, {'caller': 'predict_log_proba', 'callee': 'transform'}, {'caller': 'transform', 'callee': 'transform'}, {'caller': 'inverse_transform', 'callee': 'inverse_transform'}, {'caller': 'score', 'callee': 'transform'}], 'router': {'fit': {'sample_weight': False}, 'partial_fit': {'sample_weight': None}, 'transform': {'copy': None}, 'inverse_transform': {'copy': None}}}, 'model': {'mapping': [{'caller': 'fit', 'callee': 'fit'}, {'caller': 'fit', 'callee': 'transform'}, {'caller': 'fit', 'callee': 'fit'}, {'caller': 'predict', 'callee': 'predict'}, {'caller': 'fit_predict', 'callee': 'fit_predict'}, {'caller': 'predict_pro

## Best Practices for API Usage

Pisahkan training/test sebelum fit preprocessing, tentukan positive class dan metrik yang relevan, gunakan seed untuk pengacakan, serta simpan versi dependency. Reuse objek melalui pipeline membantu konsistensi, tetapi hasil satu split tidak menggantikan evaluasi sesuai waktu atau kelompok. Menangani custom estimator perlu mempertahankan kontrak API dan validasi input.

## Pemeriksaan Hasil

Assertion memeriksa contoh numerik dan konsistensi transformasi. Hasil clustering dinilai berdasarkan jumlah cluster, bukan kesamaan nomor label dengan cetakan buku.

In [9]:
chapter_results = {'linear_slope': float(model.coef_[0]),
 'linear_intercept': float(model.intercept_), 'linear_training_r2': float(model.score(X,y)),
 'predictions': predictions.tolist(), 'pipeline_test_accuracy': float(api_pipeline_accuracy)}
assert np.allclose(predictions, [5.75, 6.7])
assert np.allclose(X_scaled.mean(axis=0), 0)
assert len(np.unique(labels)) == 2
assert np.allclose(centerer.transform(X_scale).mean(axis=0), 0)
print(json.dumps(chapter_results, indent=2))

{
  "linear_slope": 0.9500000000000002,
  "linear_intercept": 0.04999999999999938,
  "linear_training_r2": 0.9809782608695652,
  "predictions": [
    5.75,
    6.700000000000001
  ],
  "pipeline_test_accuracy": 0.9666666666666667
}


## Ringkasan Bab

Regresi lima titik menghasilkan slope **0.95**, intercept **0.05**, dan prediksi **5,75** serta **6,70** untuk x=6 dan x=7. R-squared training sekitar **0.9810** menggambarkan kecocokan pada lima titik itu, bukan evaluasi generalisasi. KMeans menghasilkan dua kelompok; nomor cluster dapat bertukar tanpa mengubah arti pengelompokan.

StandardScaler menggunakan parameter training yang sama untuk data baru. Custom MeanCenterer menunjukkan kontrak fit/transform dan parameter hasil pembelajaran dengan suffix underscore. Pipeline Iris menghasilkan accuracy test **96.67%** pada split yang digunakan. Grid search dan randomized search memakai training/CV untuk memilih konfigurasi. Estimator tags menjelaskan kemampuan, sedangkan metadata routing mengatur penerusan informasi seperti sample_weight. Keduanya mempunyai fungsi berbeda.